# Results for model: moonshotai/kimi-k2-instruct

In [ ]:
import polars as pl
import xgboost as xgb
from pathlib import Path
from sklearn.model_selection import train_test_split

data_path = Path('./jane_street_data/train.parquet')
df = pl.read_parquet(data_path)

top_quantile = 0.85
df = df.filter(
    pl.col('feature_00') >= pl.col('feature_00')
        .quantile(top_quantile)
        .over(pl.col('date_id') // 100)
)

features = [c for c in df.columns if c.startswith('feature_')]
X = df.select(features).to_numpy()
y = df['responder_6'].to_numpy()

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)
model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=800,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    tree_method='hist'
)
model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='rmse',
    early_stopping_rounds=50,
    verbose=0
)